# Router evaluation — does the system route questions to the right paradigm? (RQ1)

The capability-boundary claim (RQ1) rests on a **router**: the dispatch decision in
`ask_anything()` decides whether a question is answerable by top-k retrieval (Bucket 1),
must be **routed away from generation** to the query layer (Buckets 2/3 — aggregate
questions retrieval would fabricate), or belongs to the **diachronic** pipeline
(Bucket 4 — change-over-time questions no k passages can carry). Until this was measured
the router was regex patterns with **no evidence**. This notebook turns the design into a
result:

- a **frozen gold question set** (80 questions, 20 per bucket, half German) → `data/`,
- routing **accuracy overall / per bucket / per language**, plus the full confusion matrix,
- the error rates that matter:
  - **fabrication risk** — Bucket-3 (content-aggregate) questions wrongly sent to retrieval;
    the system would then generate a corpus-wide claim from ~6 chunks,
  - **narrative fabrication risk** — Bucket-4 (change-over-time) questions wrongly sent to
    retrieval; the system would then narrate a corpus-wide *shift* from ~6 chunks,
  - **over-abstention** — Bucket-1 questions wrongly refused,
  - **false diachronic** — ordinary questions pulled into the keyness pipeline,
- a verbatim **failure table** (what to fix or to report as limitation),
- a report → `reports/router_evaluation_report.md`.

**Scope note.** The router separates *three* routes: retrieval, query layer, diachronic.
Separating Bucket 2 (metadata aggregate → SQL over validated columns) from Bucket 3
(content aggregate → deployed field or abstention) happens downstream in the query layer,
which knows which columns exist; that split is measured in `evaluation.ipynb`, which calls
the deployed field matcher and deny-list as well. The gold set carries 2 vs 3 labels so the
downstream split stays measurable from the same frozen questions.

## 1. Load the router from the deployed notebooks (single source of truth)
Two cells are exec'd straight out of the shipped pipeline — the Layer-1 aggregate detector
(`AGGREGATE_PATTERNS` + `aggregate_trigger`) from `rag_echr_ris.ipynb`, and the Bucket-4
change-over-time detector (`_diachronic_trigger`) from `ask.ipynb` — and then composed in
`ask_anything()`'s own dispatch order. Nothing about the router is re-implemented here, so
this evaluation can never drift from the deployed decision.

In [2]:
import json
import re
from pathlib import Path

DATA_DIR   = Path("../data")
REPORT_DIR = Path("../reports")
RAG_NB     = Path("rag_echr_ris.ipynb")
ASK_NB     = Path("ask.ipynb")
GOLD_OUT   = DATA_DIR / "router_gold_questions.csv"
REPORT_OUT = REPORT_DIR / "router_evaluation_report.md"


def exec_deployed_cell(nb_path, marker):
    """Exec the deployed cell that defines a decision function, straight out of its own
    notebook. The marker is a line the cell must contain; if the cell is ever renamed or
    moved the load fails loudly rather than silently scoring a stale copy."""
    nb = json.loads(Path(nb_path).read_text())
    for c in nb["cells"]:
        src = "".join(c["source"])
        if c["cell_type"] == "code" and marker in src:
            exec(src, globals())
            return src
    raise AssertionError(f"no cell containing {marker!r} in {nb_path}")


# Layer 1 -- aggregate detector (retrieval vs query layer).
exec_deployed_cell(RAG_NB, "AGGREGATE_PATTERNS = [")
# Bucket 4 -- change-over-time detector, added to the dispatcher after this evaluation was
# first written (which is exactly why the gold set had no Bucket-4 questions for a while).
# Its cell defines the whole diachronic handler, but nothing in it touches the corpus at
# import time (`_dia_corpus()` is lazy), so exec'ing the deployed cell whole is cheap -- and
# safer than copying the two regexes, which would go stale the day they are edited.
exec_deployed_cell(ASK_NB, "def _diachronic_trigger")
print(f"router loaded: {len(AGGREGATE_PATTERNS)} aggregate patterns from {RAG_NB.name}, "
      f"diachronic trigger from {ASK_NB.name}")

ROUTES = ["retrieval", "aggregate", "diachronic"]


def route_question(question):
    """`ask_anything()`'s dispatch, in its order, with its functions: diachronic trigger
    first, aggregate trigger second, retrieval as the fallback. (The dispatcher's first
    branch -- a string that literally starts with SELECT/WITH -- is not a natural-language
    question and is out of scope for this gold set.)

    Returns (route, the trigger that fired).
    """
    if _diachronic_trigger(question):
        return "diachronic", "change-word + time expression"
    trig = aggregate_trigger(question)
    if trig:
        return "aggregate", repr(trig)
    return "retrieval", "no trigger"

Type-1 detector ready (36 patterns)
Bucket 4 ready -- topics: parental alienation, custody, contact (+ question-derived); guard: evidence-only vocabulary, template fallback
router loaded: 36 aggregate patterns from rag_echr_ris.ipynb, diachronic trigger from ask.ipynb


## 2. Frozen gold question set — 80 questions, 20 per bucket, EN + DE
Buckets follow the RQ typology (`26.06.md`), extended with the diachronic class the
dispatcher gained later:

- **1 — explanatory / comparative / framing** → retrieval answers with citations,
- **2 — metadata aggregate** → computable by `groupby` over validated columns,
- **3 — content aggregate** → needs per-case extraction from prose; retrieval must NOT answer,
- **4 — change over time / framing shift** → *"what changed for topic T between X and Y?"*.
  No k passages carry a corpus-wide contrast, so retrieval must not answer it either; the
  faithful handler is the windowed-keyness pipeline with its faithfulness guard.

**Labelling rule for Bucket 4.** A question is gold-4 whenever the diachronic pipeline is the
right *handler* — including the German-corpus questions that pipeline answers with a reasoned
**refusal** (RIS Rechtssätze carry no reliable decision date; pre-2010 Swiss coverage is thin,
so a temporal split would compare coverage artefacts). Routing those to Bucket 1 is not a
harmless miss: generation would narrate a shift from six passages instead of refusing.

The set deliberately contains *hard* phrasings — aggregate or diachronic intent carried with
no trigger word (e.g. *"Which importance level is most common…"*, *"which terms became more
common after 2015"*, *"vor 2012 von jener nach 2013"*) — so the evaluation can fail honestly.
Questions were written against the corpus topics (contact, custody, alienation, Kindeswohl)
in each court's register.

In [4]:
# (question, gold_bucket 1|2|3, lang)
GOLD = [
    # ---- Bucket 1 — explanatory / comparative (EN) ----
    ("What positive obligations does the State have to enforce contact rights?", 1, "en"),
    ("How does the ECHR assess a child's refusal to see a parent?", 1, "en"),
    ("What role does the passage of time play in parent-child reunification cases?", 1, "en"),
    ("Under what conditions is coercive enforcement of contact against the child's will justified?", 1, "en"),
    ("How do courts weigh the best interests of the child against a parent's contact rights?", 1, "en"),
    ("What does the Court consider an effective remedy in contact-enforcement cases?", 1, "en"),
    ("When can custody be transferred to the other parent because of alienating behaviour?", 1, "en"),
    ("What is the margin of appreciation of domestic courts in custody disputes?", 1, "en"),
    ("How is expert psychological evidence treated in alienation cases?", 1, "en"),
    ("What weight is given to the child's expressed wishes in contact proceedings?", 1, "en"),
    # ---- Bucket 1 (DE) ----
    ("Wann ist von einer Vollzugsma\u00dfnahme abzusehen, wenn sie dem Kindeswohl widerspricht?", 1, "de"),
    ("Welche Voraussetzungen gelten f\u00fcr die gemeinsame Obsorge nach Trennung der Eltern?", 1, "de"),
    ("Wie begr\u00fcnden Gerichte die Einschr\u00e4nkung des Kontaktrechts?", 1, "de"),
    ("Welche Rolle spielt der Loyalit\u00e4tskonflikt des Kindes bei Kontaktrechtsentscheidungen?", 1, "de"),
    ("Unter welchen Voraussetzungen kann einem Elternteil die Obhut entzogen werden?", 1, "de"),
    ("Wie wird Bindungstoleranz in der Rechtsprechung bewertet?", 1, "de"),
    ("Was versteht die Rechtsprechung unter Kindeswohlgef\u00e4hrdung?", 1, "de"),
    ("Wann kann das Kontaktrecht gegen den Willen des Kindes durchgesetzt werden?", 1, "de"),
    ("Wie unterscheidet sich die Behandlung der Entfremdung vor dem OGH und dem EGMR?", 1, "de"),
    ("Welche Bedeutung hat die Beeinflussung des Kindes durch einen Elternteil f\u00fcr die Obsorgeentscheidung?", 1, "de"),
    # ---- Bucket 2 — metadata aggregates (EN) ----
    ("How many cases against Norway are in the corpus?", 2, "en"),
    ("Which respondent state has the most cases on contact rights?", 2, "en"),
    ("What proportion of merits judgments found a violation of Article 8?", 2, "en"),
    ("How many judgments were delivered after 2020?", 2, "en"),
    ("What is the number of communicated cases per country?", 2, "en"),
    ("How has the number of contact-rights cases changed over the years?", 2, "en"),
    ("What percentage of the corpus are admissibility decisions?", 2, "en"),
    ("Which importance level is most common among the judgments?", 2, "en"),
    ("How many Swiss decisions come from the canton of Zurich?", 2, "en"),
    ("What is the average time between communication of a case and the judgment?", 2, "en"),
    # ---- Bucket 2 (DE) ----
    ("Wie viele F\u00e4lle betreffen \u00d6sterreich?", 2, "de"),
    ("Wie hoch ist der Anteil der Verletzungsurteile?", 2, "de"),
    ("Wie oft wurde Artikel 8 verletzt?", 2, "de"),
    ("Wie viele Entscheidungen stammen aus dem Jahr 2023?", 2, "de"),
    ("Welcher Staat hat die meisten Verurteilungen?", 2, "de"),
    ("Wie hat sich die Zahl der Kontaktrechtsf\u00e4lle \u00fcber die Jahre entwickelt?", 2, "de"),
    ("Wie viele Urteile ergingen nach 2020?", 2, "de"),
    ("Wieviel Prozent der F\u00e4lle wurden f\u00fcr unzul\u00e4ssig erkl\u00e4rt?", 2, "de"),
    ("Aus welchem Kanton stammen die meisten Schweizer Entscheidungen?", 2, "de"),
    ("Wie ist die Gesamtzahl der anh\u00e4ngigen Verfahren?", 2, "de"),
    # ---- Bucket 3 — content aggregates (EN) ----
    ("How many cases involve an allegation of parental alienation?", 3, "en"),
    ("In what proportion of cases did the mother receive custody?", 3, "en"),
    ("How often do courts order supervised contact?", 3, "en"),
    ("How many children refused contact with their father across the corpus?", 3, "en"),
    ("What percentage of cases mention a psychological expert report?", 3, "en"),
    ("Do married parents win contact disputes more often than unmarried ones?", 3, "en"),
    ("How many cases involve grandparents seeking contact?", 3, "en"),
    ("What share of alienation allegations were upheld by the court?", 3, "en"),
    ("On average, how many years do contact-enforcement proceedings last?", 3, "en"),
    ("In most cases, which parent is found responsible for the alienation?", 3, "en"),
    # ---- Bucket 3 (DE) ----
    ("Wie viele Kinder leben nach der Entscheidung bei der Mutter?", 3, "de"),
    ("Wie h\u00e4ufig wird ein begleiteter Kontakt angeordnet?", 3, "de"),
    ("In welchem Anteil der F\u00e4lle wurde die Entfremdung vom Gericht festgestellt?", 3, "de"),
    ("Wie oft folgen die Gerichte dem Sachverst\u00e4ndigengutachten?", 3, "de"),
    ("Wie viele F\u00e4lle betreffen Gro\u00dfeltern als Kontaktwerber?", 3, "de"),
    ("Bekommen verheiratete Eltern in der Regel eher das Sorgerecht?", 3, "de"),
    ("Wie viele Verfahren dauerten l\u00e4nger als f\u00fcnf Jahre?", 3, "de"),
    ("Welcher Elternteil wird in den meisten F\u00e4llen f\u00fcr die Entfremdung verantwortlich gemacht?", 3, "de"),
    ("Wie hoch ist die durchschnittliche Verfahrensdauer in Kontaktrechtsf\u00e4llen?", 3, "de"),
    ("Wie oft wird das Kindeswohl als Hauptgrund f\u00fcr die Kontaktverweigerung genannt?", 3, "de"),
    # ---- Bucket 4 -- change over time / framing shift (EN) ----
    ("How has the wording around parental alienation changed between 2000 and 2020?", 4, "en"),
    ("What changed in how judgments describe custody between 2005 and 2018?", 4, "en"),
    ("How did the vocabulary of contact enforcement shift over time?", 4, "en"),
    ("Has the language used about a child's wishes evolved since 2010?", 4, "en"),
    ("What is different in alienation judgments before 2012 and after 2013?", 4, "en"),
    ("How has the framing of the child's best interests developed over time?", 4, "en"),
    ("Compare the vocabulary of contact cases in 2000-2012 with 2013-2025.", 4, "en"),
    ("Did the way the Court words coercive measures change after 2010?", 4, "en"),
    ("Which terms around reunification became more common after 2015?", 4, "en"),
    ("In what ways does the language of expert evidence differ between the early and the late judgments?", 4, "en"),
    # ---- Bucket 4 (DE) ----
    ("Wie hat sich die Formulierung zur Entfremdung in den EGMR-Urteilen zwischen 2000 und 2020 ver\u00e4ndert?", 4, "de"),
    ("Wie hat sich der Sprachgebrauch zum Kontaktrecht im Laufe der Zeit gewandelt?", 4, "de"),
    ("Was hat sich in der Wortwahl zum Kindeswohl nach 2013 ge\u00e4ndert?", 4, "de"),
    ("Wie unterscheidet sich die Sprache der Urteile vor 2012 von jener nach 2013?", 4, "de"),
    ("Hat sich die Begr\u00fcndung von Vollzugsma\u00dfnahmen seit 2010 ver\u00e4ndert?", 4, "de"),
    ("Wie hat sich die Darstellung des Loyalit\u00e4tskonflikts zwischen 2000 und 2025 entwickelt?", 4, "de"),
    ("Vergleiche die Wortwahl in Kontaktrechtsf\u00e4llen 2000-2012 mit 2013-2025.", 4, "de"),
    ("Welche Begriffe wurden nach 2015 in Obsorgeentscheidungen h\u00e4ufiger verwendet?", 4, "de"),
    ("Wie hat sich die Sprache des OGH zur Entfremdung seit 2005 ver\u00e4ndert?", 4, "de"),
    ("Wie hat sich die Formulierung in Schweizer Entscheiden seit 2010 ver\u00e4ndert?", 4, "de"),
]

import pandas as pd
gold = pd.DataFrame(GOLD, columns=["question", "bucket", "lang"])
# gold_route is the HANDLER the question belongs to -- the three routes the dispatcher
# actually chooses between. Buckets 2 and 3 share the "aggregate" route because the
# router does not separate them; the query layer does, downstream.
gold["gold_route"] = gold.bucket.map({1: "retrieval", 2: "aggregate",
                                      3: "aggregate", 4: "diachronic"})
gold.to_csv(GOLD_OUT, index=False)
print(f"gold set: {len(gold)} questions -> {GOLD_OUT.name}")
print(gold.groupby(["bucket", "lang"]).size().to_string())

gold set: 80 questions -> router_gold_questions.csv
bucket  lang
1       de      10
        en      10
2       de      10
        en      10
3       de      10
        en      10
4       de      10
        en      10


## 3. Run the router over the gold set + score
Four numbers, because the errors are not the same kind of mistake: a refused question is an
inconvenience, an aggregate or a change-over-time question that reaches generation is a
fabricated claim with citations attached.

In [6]:
_routed = [route_question(q) for q in gold.question]
gold["routed"] = [r for r, _ in _routed]
gold["trigger"] = [t for _, t in _routed]
gold["correct"] = gold.routed == gold.gold_route

acc = gold.correct.mean()
print(f"ROUTING ACCURACY (retrieval vs aggregate vs diachronic): {acc:.3f}  (n={len(gold)})\n")

print("accuracy per bucket:")
print((gold.groupby("bucket").correct.agg(["mean", "size"]).rename(
    columns={"mean": "acc", "size": "n"})).round(3).to_string())
print("\naccuracy per language:")
print((gold.groupby("lang").correct.agg(["mean", "size"]).rename(
    columns={"mean": "acc", "size": "n"})).round(3).to_string())
print("\naccuracy per bucket x language:")
print((gold.groupby(["bucket", "lang"]).correct.mean().unstack().round(3)).to_string())

print("\nconfusion matrix (rows = gold bucket, cols = route taken):")
conf = (gold.groupby(["bucket", "routed"]).size().unstack(fill_value=0)
        .reindex(columns=ROUTES, fill_value=0))
print(conf.to_string())

# --- the error rates that matter -------------------------------------------------------
b1 = gold[gold.bucket == 1]
b3 = gold[gold.bucket == 3]
b4 = gold[gold.bucket == 4]
guard = gold[gold.bucket != 1]           # everything top-k generation must not answer

fabrication_risk = (b3.routed == "retrieval").mean()
narrative_risk   = (b4.routed == "retrieval").mean()
over_abstention  = (b1.routed != "retrieval").mean()
false_diachronic = (gold[gold.bucket != 4].routed == "diachronic").mean()
generation_guard = (guard.routed != "retrieval").mean()

print(f"\nGENERATION GUARD  -- buckets 2/3/4 kept away from top-k generation: "
      f"{generation_guard:.3f} ({int((guard.routed != 'retrieval').sum())}/{len(guard)})")
print(f"FABRICATION RISK  -- Bucket-3 (content aggregate) sent to retrieval : "
      f"{fabrication_risk:.3f} ({int((b3.routed == 'retrieval').sum())}/{len(b3)})")
print(f"NARRATIVE RISK    -- Bucket-4 (change over time) sent to retrieval  : "
      f"{narrative_risk:.3f} ({int((b4.routed == 'retrieval').sum())}/{len(b4)})")
print(f"OVER-ABSTENTION   -- Bucket-1 wrongly refused                       : "
      f"{over_abstention:.3f} ({int((b1.routed != 'retrieval').sum())}/{len(b1)})")
print(f"FALSE DIACHRONIC  -- non-Bucket-4 pulled into the keyness pipeline  : "
      f"{false_diachronic:.3f} "
      f"({int((gold[gold.bucket != 4].routed == 'diachronic').sum())}/{len(gold) - len(b4)})")

# --- why n=80: design grid + what it buys statistically ---------------------------------
# 80 = 4 buckets x 2 languages x 10 questions: every bucket-language cell has equal weight,
# and 10 hand-authored questions per cell is the practical bound for deliberately *hard*,
# non-redundant items. The Wilson intervals below state what n=80 can and cannot claim;
# the gold set is a frozen CSV, so growing it never invalidates old runs -- the 60-question
# runs remain readable as the first three buckets of this one.
import math

def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, centre - half), min(1.0, centre + half))

print("\n95% Wilson CIs (what n=80 buys):")
for label, k, n in [("accuracy        ", int(gold.correct.sum()), len(gold)),
                    ("generation guard", int((guard.routed != "retrieval").sum()), len(guard)),
                    ("fabrication risk", int((b3.routed == "retrieval").sum()), len(b3)),
                    ("narrative risk  ", int((b4.routed == "retrieval").sum()), len(b4)),
                    ("over-abstention ", int((b1.routed != "retrieval").sum()), len(b1))]:
    lo, hi = wilson(k, n)
    print(f"  {label} {k}/{n} = {k/n:.3f}  CI [{lo:.2f}, {hi:.2f}]")
print("  <- per-bucket n=20 is the binding constraint on every rate but the headline")

ROUTING ACCURACY (retrieval vs aggregate vs diachronic): 0.812  (n=80)

accuracy per bucket:
         acc   n
bucket          
1       1.00  20
2       0.95  20
3       0.90  20
4       0.40  20

accuracy per language:
        acc   n
lang           
de    0.725  40
en    0.900  40

accuracy per bucket x language:
lang     de   en
bucket          
1       1.0  1.0
2       1.0  0.9
3       0.9  0.9
4       0.0  0.8

confusion matrix (rows = gold bucket, cols = route taken):
routed  retrieval  aggregate  diachronic
bucket                                  
1              20          0           0
2               1         19           0
3               2         18           0
4              12          0           8

GENERATION GUARD  -- buckets 2/3/4 kept away from top-k generation: 0.750 (45/60)
FABRICATION RISK  -- Bucket-3 (content aggregate) sent to retrieval : 0.100 (2/20)
NARRATIVE RISK    -- Bucket-4 (change over time) sent to retrieval  : 0.600 (12/20)
OVER-ABSTENTION   -- Bucke

## 4. Failure table — every misrouted question, verbatim
These are the thing to act on. Severity follows the direction of the error: a Bucket-3 or
Bucket-4 question that reached retrieval would be answered by generation from a handful of
passages (a fabricated statistic, or a fabricated narrative of change); an over-abstention
is a lost answer, which the router is *biased towards* by design.

In [8]:
_SEVERITY = {
    (3, "retrieval"):  "FABRICATION RISK",     # corpus-wide statistic from ~6 passages
    (4, "retrieval"):  "NARRATIVE FABRICATION",  # corpus-wide change narrated from ~6 passages
    (2, "retrieval"):  "missed aggregate",
    (3, "diachronic"): "wrong query path",
    (2, "diachronic"): "wrong query path",
    (4, "aggregate"):  "wrong query path",
    (1, "aggregate"):  "over-abstention",
    (1, "diachronic"): "over-abstention",
}

fails = gold[~gold.correct].copy()
fails["severity"] = [_SEVERITY.get((b, r), "misroute")
                     for b, r in zip(fails.bucket, fails.routed)]
if fails.empty:
    print("no routing failures on the gold set")
else:
    _ord = {"NARRATIVE FABRICATION": 0, "FABRICATION RISK": 1, "missed aggregate": 2,
            "wrong query path": 3, "over-abstention": 4, "misroute": 5}
    for _, r in fails.sort_values("severity", key=lambda s: s.map(_ord)).iterrows():
        print(f"[{r.severity:21s}] bucket={r.bucket} lang={r.lang} routed={r.routed}")
        print(f"    Q: {r.question}")
        print(f"    trigger: {r.trigger}")
print(f"\n{len(fails)} failures / {len(gold)} questions")
print(f"failures by severity: "
      f"{fails.severity.value_counts().to_dict() if not fails.empty else {}}")

[NARRATIVE FABRICATION] bucket=4 lang=en routed=retrieval
    Q: Which terms around reunification became more common after 2015?
    trigger: no trigger
[NARRATIVE FABRICATION] bucket=4 lang=en routed=retrieval
    Q: In what ways does the language of expert evidence differ between the early and the late judgments?
    trigger: no trigger
[NARRATIVE FABRICATION] bucket=4 lang=de routed=retrieval
    Q: Wie hat sich die Formulierung zur Entfremdung in den EGMR-Urteilen zwischen 2000 und 2020 verändert?
    trigger: no trigger
[NARRATIVE FABRICATION] bucket=4 lang=de routed=retrieval
    Q: Wie hat sich der Sprachgebrauch zum Kontaktrecht im Laufe der Zeit gewandelt?
    trigger: no trigger
[NARRATIVE FABRICATION] bucket=4 lang=de routed=retrieval
    Q: Was hat sich in der Wortwahl zum Kindeswohl nach 2013 geändert?
    trigger: no trigger
[NARRATIVE FABRICATION] bucket=4 lang=de routed=retrieval
    Q: Wie unterscheidet sich die Sprache der Urteile vor 2012 von jener nach 2013?
    tri

## 5. Report → `reports/router_evaluation_report.md`

In [10]:
n_b = gold.groupby("bucket").size()
lines = ["# Router evaluation report (RQ1 - capability boundary)\n\n"]
lines.append(f"- gold set: **{len(gold)}** questions (frozen: `{GOLD_OUT.name}`), "
             f"{n_b.min()} per bucket across {len(n_b)} buckets, "
             f"{int((gold.lang == 'de').sum())} German / {int((gold.lang == 'en').sum())} English\n"
             f"- router: the deployed dispatch decision, exec'd from the shipped notebooks and "
             f"composed in `ask_anything()`'s order - `_diachronic_trigger` (`ask.ipynb`) then "
             f"`aggregate_trigger` ({len(AGGREGATE_PATTERNS)} patterns, `rag_echr_ris.ipynb`), "
             f"retrieval as the fallback\n\n")
lines.append(f"## Headline numbers\n"
             f"- routing accuracy (3 routes: retrieval / aggregate / diachronic): **{acc:.3f}**\n"
             f"- generation guard (buckets 2/3/4 kept away from top-k generation): "
             f"**{generation_guard:.3f}** "
             f"({int((guard.routed != 'retrieval').sum())}/{len(guard)})\n"
             f"- fabrication risk (Bucket-3 sent to retrieval): **{fabrication_risk:.3f}** "
             f"({int((b3.routed == 'retrieval').sum())}/{len(b3)})\n"
             f"- narrative fabrication risk (Bucket-4 sent to retrieval): **{narrative_risk:.3f}** "
             f"({int((b4.routed == 'retrieval').sum())}/{len(b4)})\n"
             f"- over-abstention (Bucket-1 refused): **{over_abstention:.3f}** "
             f"({int((b1.routed != 'retrieval').sum())}/{len(b1)})\n"
             f"- false diachronic (non-Bucket-4 into the keyness pipeline): "
             f"**{false_diachronic:.3f}**\n\n")

lines.append("## Accuracy per bucket x language\n\n")
tbl = gold.groupby(["bucket", "lang"]).correct.mean().unstack().round(3)
cols = list(tbl.columns)
lines.append("| bucket | " + " | ".join(cols) + " |\n")
lines.append("|" + "---|" * (len(cols) + 1) + "\n")
for b, row in tbl.iterrows():
    lines.append(f"| {b} | " + " | ".join(str(row[c]) for c in cols) + " |\n")
lines.append("\n")

lines.append("## Confusion matrix (rows = gold bucket, cols = route taken)\n\n")
lines.append("| gold bucket | " + " | ".join(ROUTES) + " |\n")
lines.append("|" + "---|" * (len(ROUTES) + 1) + "\n")
for b, row in conf.iterrows():
    lines.append(f"| {b} | " + " | ".join(str(int(row[c])) for c in ROUTES) + " |\n")
lines.append("\n")

lines.append("## Failures (verbatim)\n\n")
if fails.empty:
    lines.append("none\n")
else:
    _ord = {"NARRATIVE FABRICATION": 0, "FABRICATION RISK": 1, "missed aggregate": 2,
            "wrong query path": 3, "over-abstention": 4, "misroute": 5}
    for _, r in fails.sort_values("severity", key=lambda s: s.map(_ord)).iterrows():
        lines.append(f"- **{r.severity}** (bucket {r.bucket}, {r.lang}): \u201c{r.question}\u201d\n")
lines.append("\n")

lo_a, hi_a = wilson(int(gold.correct.sum()), len(gold))
lo_f, hi_f = wilson(int((b3.routed == "retrieval").sum()), len(b3))
lo_n, hi_n = wilson(int((b4.routed == "retrieval").sum()), len(b4))
lines.append(f"## Why n={len(gold)} (design + statistics)\n"
             f"- {len(gold)} = {len(n_b)} buckets x 2 languages x {int(n_b.min() / 2)} "
             f"hand-authored questions: equal-weight cells; {int(n_b.min() / 2)} non-redundant "
             f"*hard* items per cell is the practical authoring bound. Bucket 4 "
             f"(change over time) was added when the dispatcher gained its diachronic branch - "
             f"until then a whole shipped capability was unmeasured, and the questions that "
             f"exercise it were scored as if they belonged to another bucket.\n"
             f"- What n={len(gold)} buys (95% Wilson): accuracy {acc:.2f} -> CI "
             f"[{lo_a:.2f}, {hi_a:.2f}]; fabrication risk {fabrication_risk:.2f} at per-bucket "
             f"n={len(b3)} -> CI [{lo_f:.2f}, {hi_f:.2f}]; narrative fabrication risk "
             f"{narrative_risk:.2f} -> CI [{lo_n:.2f}, {hi_n:.2f}] - stated, not hidden: the "
             f"headline is tight, per-bucket rates are indicative. The gold set is a frozen "
             f"CSV; it can be grown without invalidating prior runs.\n")

lines.append("\n## Reading\n"
             "- The router chooses between three handlers; Bucket 2 vs 3 is separated "
             "downstream by the query layer (which knows the validated columns) and is scored "
             "in `evaluation.ipynb` over these same frozen questions. What this report "
             "measures is the boundary that carries the faithfulness claim: **no aggregate and "
             "no change-over-time question may reach generation**.\n"
             "- Bucket 4 is labelled by handler, not by answerability: a German-corpus change "
             "question is gold-4 because the diachronic branch is what *refuses* it, with the "
             "stated corpus facts (undatable RIS Rechtssaetze, thin pre-2010 Swiss coverage). "
             "Sending it to retrieval instead would narrate a shift from six passages.\n"
             "- Failures where the intent carries no lexical trigger (e.g. \u201cmost common\u201d, "
             "\u201cmore often than\u201d, \u201cbecame more common after 2015\u201d) are the known limit "
             "of a pattern router - candidates for a semantic-routing extension, or reported as "
             "a stated limitation.\n")
if narrative_risk > 0:
    _de_miss = int(((b4.lang == "de") & (b4.routed == "retrieval")).sum())
    lines.append(f"- **Asymmetry worth naming:** `aggregate_trigger` carries German patterns; "
                 f"`_DIA_CHANGE`/`_DIA_TIME` do not. {_de_miss} of the "
                 f"{int((b4.lang == 'de').sum())} German Bucket-4 questions therefore miss the "
                 f"diachronic branch entirely. This is a lexicon gap in one deployed regex, not "
                 f"a property of the approach - it is reported here rather than patched "
                 f"silently, because the fix changes the headline number.\n")

REPORT_OUT.write_text("".join(lines), encoding="utf-8")
print("wrote", REPORT_OUT)

wrote ../reports/router_evaluation_report.md
